## Subscription Behaviors 

This project analyzes subscription renewals for a SaaS company by integrating customer, subscription, and economic data to understand how renewal rates vary across client segments and to describe the economic conditions present at the time of renewal events.

Three datasets were provided:

`client_details.csv`:

`client_id`: Unique identifier for each client.\
`company_size`: Size of the company (Small, Medium, Large).\
`industry`: Industry to which the client belongs (Fintech, Gaming, Crypto, AI, E-commerce)\
`location`:  Location of the client (New York, New Jersey, Pennsylvania, Massachusetts.

`subscription_records.csv`:

`client_id`: Unique identifier for each client. \
`subscription_type`: Type of subscription (Yearly, Monthly). \
`start_date`: Start date of the subscription - YYYY-MM-DD. \
`end_date`: End date of the subscription - YYYY-MM-DD.\
`renewed`: Indicates whether the subscription was renewed (True, False)

`economic_indicators.csv`: 
`start_date`: Start date of the economic indicator (Quarterly) - YYYY-MM-DD.\
`end_date`: End date of the economic indicator (Quarterly) - YYYY-MM-DD.\
`inflation_rate`: Inflation rate in the period.\
`gdp_growth_rate`: Gross Domestic Product (GDP) growth rate in the period.

The datasets were merged using client_id to link client details with subscription records, and merge_asof was used to align each subscription end_date with the corresponding economic indicators (inflation and GDP growth) based on time. The analysis focused on identifying the total number of Fintech and Crypto clients, determining which industry has the highest subscription renewal rate, and calculating the average inflation rate at the time of renewal for clients who renewed their subscriptions.

Overall, the results show how renewal rates differ across industries and describe the economic conditions present during renewal events. The findings provide descriptive insights into customer behavior and the macroeconomic environment at the time of renewals.

In [48]:
# Import required libraries
import pandas as pd

# Import data and convert date columns from string to datetime
client_details = pd.read_csv('data/client_details.csv')

subscription_records = pd.read_csv('data/subscription_records.csv', 
                                   parse_dates = ['start_date','end_date'])

economic_indicators = pd.read_csv('data/economic_indicators.csv', 
                                  parse_dates = ['start_date','end_date'])

In [49]:
# 1. How many total Fintech and Crypto clients do they have?
total_fintech_crypto_clients = (client_details[client_details['industry']
                               .isin(['Fintech', 'Crypto'])].shape[0])

total_fintech_crypto_clients

47

In [50]:
# 2. Which industry has the highest renewal rate?

# Merge the client_details and subscription_records
client_subs = (client_details
               .merge(subscription_records, 
                      on = 'client_id', 
                      how = 'left'))

# Group by industry and find the renewal rate
industry_renewal_rate = (client_subs
                         .groupby('industry')['renewed']
                         .agg('mean')
                         .sort_values(ascending = False))
# Assign value to top_industry
top_industry = industry_renewal_rate.index[0]

top_industry

'Gaming'

In [51]:
# 3. For clients who renewed their subscriptions, what was the average inflation rate when their subscriptions were renewed?

# Sort subscription_records by ascending end_date
subs_sorted = subscription_records.sort_values(by = 'end_date', ascending = True).reset_index(drop=True)

# Sort economic_indicators by ascending start_date
econ_sorted = economic_indicators.sort_values(by = 'start_date', ascending = True).reset_index(drop=True)

# Merge the sorted susbcription_type end_date(renewal period) with the nearest sorted economic_indicators start_date(when inflation began)
sub_econ_merged = pd.merge_asof(subs_sorted,econ_sorted, 
                                left_on = 'end_date', 
                                right_on = 'start_date', 
                                direction = 'backward')

# Get the clients that renewed and find the average inflation rate when their subscription was renewed
average_inflation_for_renewals = sub_econ_merged[sub_econ_merged['renewed']==True]['inflation_rate'].mean()

average_inflation_for_renewals

4.418909090909092